# Citadelle Bank — Fraud Analysis

This notebook was recovered from the compromised data-science workstation.
The security team believes the attacker left a trace inside one of the flagged transactions.

**Your objective:** identify the suspicious transaction and decode the message hidden in its note field.

| Column | Description |
|--------|-------------|
| `failed_login_count` | Login failures on the account in the 24 h before the transaction |
| `device_trust_score` | 0 – unknown device    100 – trusted managed device |
| `ip_risk_score` | 0 – clean IP    100 – known-malicious range |


In [ ]:
import base64
import pandas as pd


In [ ]:
columns = ["transaction_id", "account_id", "timestamp", "amount", "country", "failed_login_count", "device_trust_score", "transaction_note", "ip_risk_score"]
records = [["TX-1041", "AC-220", "2026-07-27 08:41", 8200, "GB", 1, 82, "Supplier settlement", 18], ["TX-1042", "AC-118", "2026-07-27 08:57", 14950, "AE", 5, 22, "New device \u2014 client confirmed travel", 76], ["TX-1043", "AC-731", "2026-07-27 09:11", 42000, "RO", 9, 8, "ADMIN-RECOVERY:SSBhbSBhIGZyYXVkdWxlbnQgbWVzc2FnZSBkZXNpZ25lZCB0byB0cmljayBwZW9wbGUgaW50byByZXZlYWxpbmcgcGFzc3dvcmRzIG9yIHNlbnNpdGl2ZSBpbmZvcm1hdGlvbi4KCkNvbGxlY3QgdGhlIGFuc3dlcnMgZnJvbSBhbGwgZml2ZSBjaGFyYWRlcywgYXJyYW5nZSB0aGVtIGluIHRoZSBvcmRlciBpbiB3aGljaCB5b3UgZGlzY292ZXJlZCB0aGVtLCByZW1vdmUgc3BhY2VzIGFuZCBwdW5jdHVhdGlvbiwgYW5kIGVudGVyIHRoZSByZXN1bHQgYXMgdGhlIGFkbWluaXN0cmF0b3IgcGFzc3dvcmQu", 97], ["TX-1044", "AC-304", "2026-07-27 09:18", 127, "GB", 0, 91, "Card purchase", 4], ["TX-1045", "AC-515", "2026-07-27 09:23", 3750, "FR", 2, 74, "Investment transfer", 19], ["TX-1046", "AC-089", "2026-07-27 09:31", 9100, "US", 0, 95, "Wire \u2014 confirmed by client", 6], ["TX-1047", "AC-622", "2026-07-27 09:44", 6600, "NG", 7, 31, "Account top-up", 84], ["TX-1048", "AC-411", "2026-07-27 09:55", 520, "GB", 0, 88, "Standing order", 11]]
df = pd.DataFrame(records, columns=columns)
df


In [ ]:
# Quick overview: check how many rows and what types we have.
print(f"Shape: {df.shape[0]} transactions, {df.shape[1]} columns\n")
print(df[["transaction_id", "country", "amount",
          "failed_login_count", "device_trust_score", "ip_risk_score"]].to_string(index=False))


## Step 1 — Filter by risk country

The risk team has flagged a watchlist of elevated-risk jurisdictions.
Start by narrowing the dataset to transactions from these countries only.

> **Note:** A high country-risk score alone is not proof of malicious activity.
> One of the candidates has a legitimate explanation in its note field.


In [ ]:
watchlist_countries = ["RO", "AE", "NG", "UA"]
candidates = df[df["country"].isin(watchlist_countries)]
candidates[["transaction_id", "country", "failed_login_count",
            "device_trust_score", "ip_risk_score", "transaction_note"]]


## Step 2 — Apply multi-signal risk thresholds

Multiple weak signals together are much stronger evidence than any single indicator.
Apply **all three** thresholds simultaneously to narrow down to the most critical record:

- **failed_login_count ≥ 8** — nine or more consecutive failures suggest a brute-force attempt
- **device_trust_score < 15** — below 15 means an unrecognised, unmanaged device
- **ip_risk_score > 90** — above 90 places the source in a known-bad IP range

> **Verification tip:** After filtering, look at which candidates were cleared and why.
> Do not decode anything until you can explain why each ruled-out record was legitimate.


In [ ]:
# All three thresholds must be satisfied simultaneously.
suspicious = candidates[
    (candidates["failed_login_count"] >= 8) &
    (candidates["device_trust_score"] < 15) &
    (candidates["ip_risk_score"] > 90)
]
suspicious


## Step 3 — Decode the embedded clue

The suspicious transaction note contains a prefixed Base64 message.
Strip the prefix (`ADMIN-RECOVERY:`) and decode the remainder to recover
the final investigation clue.


In [ ]:
# Decode the clue embedded in the suspicious transaction note.
note = suspicious.iloc[0]["transaction_note"]
encoded = note.split(":", 1)[1]  # text after "ADMIN-RECOVERY:"
print(base64.b64decode(encoded).decode())
